# Nemotron LoRA SFT — Colab Training Notebook

**Requirements:** A100 80 GB runtime (Colab Pro+). The 30B bf16 model is ~60 GB.

Steps: GPU check → Mount Drive → Install packages → HF token → Download model → Train → Package

## 0. Clone repo from GitHub
Runs once; skipped on restart if already present.

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Axentalan-VI/nemotron-reasoning.git"
REPO_DIR = Path("/content/nemotron")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    print("Cloned to", REPO_DIR)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    print("Repo already present — pulled latest.")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Contents:", [p.name for p in sorted(REPO_DIR.iterdir())])


In [ ]:
import json
from pathlib import Path

REPO_DIR = Path("/content/nemotron")
TRACES   = REPO_DIR / "data" / "distill" / "teacher_traces.jsonl"
OUTPUT   = REPO_DIR / "outputs" / "colab_run"
OUTPUT.mkdir(parents=True, exist_ok=True)

if not TRACES.exists():
    raise FileNotFoundError(
        f"Traces not found: {TRACES}. "
        "Upload teacher_traces.jsonl via: "
        "from google.colab import files; files.upload()"
    )

kept  = sum(1 for l in TRACES.open() if l.strip() and json.loads(l).get("kept"))
total = sum(1 for l in TRACES.open() if l.strip())
print(f"Traces: {total} rows, {kept} kept ({kept/total*100:.1f}%)")
print(f"Output: {OUTPUT}")


## 1. GPU check
Must be **A100 80 GB** or H100. Change via: Runtime → Change runtime type.

In [ ]:
import subprocess
res = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True)
info = res.stdout.strip()
print(info)
try:
    vram = int(info.split(',')[1].strip().split()[0])
    print('OK' if vram >= 70000 else 'WARNING: need ~80 GB',
          f'— {vram/1024:.0f} GB VRAM')
except Exception:
    pass


## 3. Install packages
Adds HF/TRL stack on top of Colab's PyTorch. No bitsandbytes — bnb 4-bit is incompatible with Nemotron-H Mamba layers; we train in plain bf16.

In [ ]:
%%capture
# mamba-ssm and causal-conv1d compile CUDA kernels (~5 min)
!pip install -q mamba-ssm causal-conv1d
!pip install -q "transformers>=4.45" "peft>=0.12" "trl>=0.11" "accelerate>=0.34" "datasets>=2.20" safetensors sentencepiece python-dotenv


In [ ]:
import pkg_resources
for pkg in ['torch', 'transformers', 'peft', 'trl', 'accelerate', 'datasets', 'safetensors']:
    try:
        v = pkg_resources.get_distribution(pkg).version
        print(f'  {pkg:<16s} {v}')
    except Exception:
        print(f'  {pkg:<16s} NOT FOUND')


## 4. HuggingFace token
Add `HF_TOKEN` in Colab Secrets (key icon in left sidebar), or paste the value below.

In [ ]:
import os
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets.")
except Exception:
    HF_TOKEN = ""  # <- paste token here if not using Secrets
if not HF_TOKEN:
    raise ValueError("HF_TOKEN empty. Set it in Colab Secrets or paste it above.")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
print("Token set.")


## 5. Download base model
~60 GB. Cached to Drive so Colab restarts skip this. Update `MODEL_ID` if the competition uses a different HF repo.

In [ ]:
# ---- EDIT IF NEEDED ----
MODEL_ID  = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
MODEL_DIR = REPO_DIR / "model_cache" / "nemotron-30b"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
# ------------------------

# Check for actual safetensors shards (not just .metadata stubs)
real_shards = [p for p in MODEL_DIR.rglob("*.safetensors") if p.stat().st_size > 1_000_000]
print(f"Real shards found: {len(real_shards)} / 13")

if len(real_shards) < 13:
    from huggingface_hub import snapshot_download
    print(f"Downloading {MODEL_ID} (30-60 min)...")
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=str(MODEL_DIR),
        token=HF_TOKEN,
        ignore_patterns=["*.msgpack", "*.h5", "flax_model*", "tf_model*"],
        resume_download=True,
    )
    print("Download complete.")
else:
    print("All shards present — skipping download.")

tok_files = [p.name for p in MODEL_DIR.iterdir() if "token" in p.name.lower()]
print("Tokenizer files:", tok_files)


## 6. Training config
Defaults: rank=32, alpha=64, 2 epochs, lr=2e-4, effective batch=16.

In [ ]:
TRAIN_CONFIG = dict(
    base_model    = str(MODEL_DIR),
    traces        = str(TRACES),
    output_dir    = str(OUTPUT),
    epochs        = 2.0,
    batch_size    = 1,
    grad_accum    = 16,
    lr            = 2e-4,
    max_seq_len   = 4096,
    rank          = 32,
    alpha         = 64,
    dropout       = 0.05,
    warmup_ratio  = 0.03,
    save_steps    = 100,
    logging_steps = 10,
    seed          = 42,
    report_to     = 'none',
    run_name      = 'colab_run',
)
for k, v in TRAIN_CONFIG.items():
    print(f'  {k:<16s} = {v}')


## 7. Run SFT training
Adapter saved to `outputs/colab_run/adapter/` on Drive. Estimate: 2–4 hours on A100 80 GB with 1,121 examples × 2 epochs.

In [ ]:
import sentencepiece, tiktoken
print(sentencepiece.__version__, tiktoken.__version__)

In [ ]:
import os, sys
os.chdir(str(REPO_DIR))

from src.train.sft_qlora import main as sft_main

sys.argv = [
    'sft_qlora',
    '--base-model',    TRAIN_CONFIG['base_model'],
    '--traces',        TRAIN_CONFIG['traces'],
    '--output-dir',    TRAIN_CONFIG['output_dir'],
    '--epochs',        str(TRAIN_CONFIG['epochs']),
    '--batch-size',    str(TRAIN_CONFIG['batch_size']),
    '--grad-accum',    str(TRAIN_CONFIG['grad_accum']),
    '--lr',            str(TRAIN_CONFIG['lr']),
    '--max-seq-len',   str(TRAIN_CONFIG['max_seq_len']),
    '--rank',          str(TRAIN_CONFIG['rank']),
    '--alpha',         str(TRAIN_CONFIG['alpha']),
    '--dropout',       str(TRAIN_CONFIG['dropout']),
    '--warmup-ratio',  str(TRAIN_CONFIG['warmup_ratio']),
    '--save-steps',    str(TRAIN_CONFIG['save_steps']),
    '--logging-steps', str(TRAIN_CONFIG['logging_steps']),
    '--seed',          str(TRAIN_CONFIG['seed']),
    '--report-to',     TRAIN_CONFIG['report_to'],
    '--run-name',      TRAIN_CONFIG['run_name'],
]
sft_main()


In [ ]:
from pathlib import Path
model_dir = Path(TRAIN_CONFIG['base_model'])
tok_files = [p.name for p in model_dir.iterdir() if 'token' in p.name.lower()]
print("Tokenizer files:", tok_files)

# Check if sentencepiece is importable in this kernel
try:
    import sentencepiece
    print("sentencepiece importable:", sentencepiece.__version__)
except ImportError:
    print("sentencepiece NOT importable — needs kernel restart")


In [ ]:
from pathlib import Path
model_dir = Path(TRAIN_CONFIG['base_model'])
files = list(model_dir.rglob('*'))
if not files:
    print("Model directory is EMPTY")
else:
    for p in sorted(files)[:30]:
        size = p.stat().st_size / 1e6 if p.is_file() else 0
        print(f"  {p.relative_to(model_dir)}  ({size:.1f} MB)")
    print(f"\nTotal files: {len(files)}")


In [ ]:
# Find the actual model path in HF cache
import glob
snapshots = glob.glob(os.path.expanduser('~/.cache/huggingface/hub/models--nvidia*/snapshots/*/'))
print("Found snapshots:", snapshots)

# Use the first one found
if snapshots:
    TRAIN_CONFIG['base_model'] = snapshots[0].rstrip('/')
    print("Using:", TRAIN_CONFIG['base_model'])
    
    # Verify tokenizer files
    tok_files = [Path(TRAIN_CONFIG['base_model'] + '/' + p).name 
                 for p in os.listdir(TRAIN_CONFIG['base_model']) if 'token' in p.lower()]
    print("Tokenizer files:", tok_files)


In [ ]:
# Pull the fix
import subprocess
subprocess.run(["git", "-C", "/content/nemotron", "pull", "--ff-only"], check=True)


In [ ]:
# Reload the module so the change takes effect in this session
import importlib, src.train.sft_qlora
importlib.reload(src.train.sft_qlora)
from src.train.sft_qlora import main as sft_main
print("Reloaded — use_fast=False")


In [ ]:
import os, sys
from pathlib import Path

# Ensure REPO_DIR is on the path regardless of chdir
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

os.chdir(str(REPO_DIR))

# Verify src is findable
src_path = Path(REPO_DIR) / 'src'
assert src_path.exists(), f'src/ not found at {src_path} — check upload'
print('src/ found:', src_path)

from src.train.sft_qlora import main as sft_main
print('Import OK')


## 8. Verify adapter

In [ ]:
import json
from pathlib import Path
adapter_dir = Path(TRAIN_CONFIG['output_dir']) / 'adapter'
cfg = json.loads((adapter_dir / 'adapter_config.json').read_text())
r = cfg.get('r', -1)
print(f'LoRA rank  : {r}  ({"OK" if r <= 32 else "OVER LIMIT"})')
print(f'alpha      : {cfg.get("lora_alpha")}')
print(f'targets    : {cfg.get("target_modules")}')
print()
for p in sorted(adapter_dir.iterdir()):
    print(f'  {p.name:<40s}  {p.stat().st_size/1e6:>8.2f} MB')


## 9. Package submission.zip

In [ ]:
from pathlib import Path
from src.submit.build_submission import build
adapter_dir     = Path(TRAIN_CONFIG['output_dir']) / 'adapter'
submission_path = REPO_DIR / 'outputs' / 'submission.zip'
report = build(adapter_dir, submission_path)
print(report.summary())
print(f'\n-> {submission_path}  ({submission_path.stat().st_size/1e6:.2f} MB)')
print('Ready to submit on Kaggle.')
